# Loan Sanction Prediction — End-to-End Data Science Project

## Project Overview

This project focuses on analyzing historical loan application data and building a machine learning model to predict whether a loan application will be approved.

The project will cover the complete data science workflow:

1. Data understanding
2. Data cleaning
3. Exploratory data analysis (EDA)
4. Feature engineering
5. SQL analysis
6. Machine learning model development
7. Model evaluation and comparison
8. Prediction on unseen data
9. Model interpretation
10. Power BI dashboard
11. Project documentation for GitHub

### Dataset

The dataset is divided into two files:

- `loan_sanction_train.csv` — historical loan applications containing the target variable `Loan_Status`.
- `loan_sanction_test.csv` — loan applications without the target variable, which will be used for final predictions.

### Machine Learning Objective

The objective is to build a supervised machine learning classification model that predicts whether a loan application will be approved (`Y`) or rejected (`N`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings('ignore')

train = pd.read_csv("C:/Users/user/Desktop/Loan Application/Data/loan_sanction_train.csv")
test = pd.read_csv("C:/Users/user/Desktop/Loan Application/Data/loan_sanction_test.csv")

In [ ]:
train.shape

In [ ]:
test.shape

In [ ]:
train.head()

In [ ]:
test.head()

## 1. Dataset Structure

The training dataset contains historical loan applications along with their loan approval status.

The test dataset contains similar applicant information but does not contain `Loan_Status`. This allows the trained machine learning model to generate predictions for unseen applications.

The training dataset contains 614 records and 13 columns, while the test dataset contains 367 records and 12 columns.

In [ ]:
train.columns

## 2. Target Variable

The target variable for this project is `Loan_Status`.

It represents the final loan decision:

- `Y` — Loan approved
- `N` — Loan rejected

The remaining applicant characteristics will be used as predictor variables to train the machine learning models.

In [ ]:
train.info()

In [ ]:
train.describe()

In [ ]:
train.isna().sum()

## 3. Missing Value Investigation

The training dataset contains missing values in several predictor variables. Before imputing these values, the missing records will be investigated to understand their distribution and determine an appropriate treatment strategy.

The target variable, `Loan_Status`, contains no missing values, which means all training records have a known outcome.

In [ ]:
missing_rows = train[train.isna().any(axis=1)]

missing_rows.head()

In [ ]:
missing_rows.shape

In [ ]:
missing_rows.isna().sum().sort_values(ascending=False)

In [ ]:
missing_rows.isna().sum(axis=1).value_counts().sort_index()

In [ ]:
categorical_cols = [
    'Gender',
    'Married',
    'Dependents',
    'Education',
    'Self_Employed',
    'Property_Area'
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(train[col].value_counts(dropna=False))

In [ ]:
numerical_cols = [
    'ApplicantIncome',
    'CoapplicantIncome',
    'LoanAmount',
    'Loan_Amount_Term',
    'Credit_History'
]

train[numerical_cols].describe()

## 4. Missing Values and Loan Approval

Before imputing missing values, we investigate whether the absence of information is associated with the loan approval outcome.

This helps determine whether missingness is random or whether it may contain useful information about the applicant.

The target variable `Loan_Status` is used only for investigation at this stage. No imputation or model training is performed yet.

In [ ]:
missing_cols = [
    'Gender',
    'Married',
    'Dependents',
    'Self_Employed',
    'LoanAmount',
    'Loan_Amount_Term',
    'Credit_History'
]

for col in missing_cols:
    print(f"\n--- {col} ---")
    
    missing = train[train[col].isna()]
    not_missing = train[train[col].notna()]
    
    print("Missing:", len(missing))
    print("Missing approval rate:")
    print(missing['Loan_Status'].value_counts(normalize=True))
    
    print("\nNot missing approval rate:")
    print(not_missing['Loan_Status'].value_counts(normalize=True))

In [ ]:
pd.crosstab(
    train['Credit_History'],
    train['Loan_Status'],
    normalize='index'
)

## 5. Numerical Variable Investigation

The numerical variables are examined before imputation to identify skewness, unusual values, and the most appropriate measure for replacing missing observations.

Median-based imputation may be preferable for highly skewed variables because extreme values can strongly influence the mean.

In [ ]:
train[['ApplicantIncome',
       'CoapplicantIncome',
       'LoanAmount',
       'Loan_Amount_Term']].describe()

In [ ]:
train['LoanAmount'].value_counts().head(20)

In [ ]:
train['Loan_Amount_Term'].value_counts(dropna=False).sort_index()

In [ ]:
train[train['LoanAmount'].isna()][
    ['ApplicantIncome',
     'CoapplicantIncome',
     'LoanAmount',
     'Loan_Status']
]

In [ ]:
train.groupby(train['LoanAmount'].isna())[
    ['ApplicantIncome', 'CoapplicantIncome']
].median()

In [ ]:
train['Loan_Amount_Term'].value_counts(dropna=False).sort_index()

In [ ]:
train['Loan_Status'].value_counts()

In [ ]:
train['Loan_Status'].value_counts(normalize=True)

In [ ]:
train['Loan_Amount_Term'].value_counts(dropna=False).sort_index()

In [ ]:
train[train['LoanAmount'].isna()][
    ['ApplicantIncome',
     'CoapplicantIncome',
     'LoanAmount',
     'Loan_Status']
]

In [ ]:
train.groupby(train['LoanAmount'].isna())[
    ['ApplicantIncome', 'CoapplicantIncome']
].median()

In [ ]:
train['Loan_Status'].value_counts(normalize=True)

In [ ]:
train['Loan_Amount_Term'].value_counts(dropna=False).sort_index()

In [ ]:
train['Credit_History'].value_counts(dropna=False)

## 6. Missing Value Treatment Strategy

After investigating the missing values and their relationship with the target variable, the following treatment strategy was selected:

### Categorical Variables

- `Gender` → impute with the mode.
- `Married` → impute with the mode.
- `Dependents` → impute with the mode.
- `Self_Employed` → impute with the mode.

These variables have relatively small proportions of missing observations, and their missing groups do not show strong evidence that missingness represents a distinct category.

### Numerical Variables

- `LoanAmount` → impute with the median.

The distribution of loan amounts is affected by high values, making the median more robust than the mean.

- `Loan_Amount_Term` → impute with the mode.

The value `360` is overwhelmingly the most common loan term, so it is more appropriate than using the arithmetic mean.

### Credit History

`Credit_History` will not be imputed directly with `0`.

Because missing credit history has a very different approval pattern from an observed value of `0`, missing credit history will initially be preserved as a separate category so that the machine learning model can learn its effect.

This approach avoids incorrectly assuming that missing credit history means no credit history.

In [ ]:
df = train.copy()
df.shape

## 7. Handling Missing Categorical Values

The categorical variables `Gender`, `Married`, `Dependents`, and `Self_Employed` contain a relatively small number of missing observations.

Since their missingness does not provide strong evidence of a separate category, the missing values will be replaced using the mode of each respective variable.

In [ ]:
categorical_impute = [
    'Gender',
    'Married',
    'Dependents',
    'Self_Employed'
]

for col in categorical_impute:
    df[col] = df[col].fillna(df[col].mode()[0])

In [ ]:
df[categorical_impute].isna().sum()

## 8. Handling Missing Loan Amounts

The `LoanAmount` variable contains 22 missing observations.

Because loan amounts can be affected by extreme values, the median is used instead of the mean. Median imputation is more robust to skewed distributions and extreme observations.

In [ ]:
df['LoanAmount'] = df['LoanAmount'].fillna(
    df['LoanAmount'].median()
)

In [ ]:
df['LoanAmount'].isna().sum()

## 9. Handling Missing Loan Term

The `Loan_Amount_Term` variable contains 14 missing observations.

The value `360` is by far the most common loan term in the dataset. Therefore, the missing values are replaced using the mode rather than the mean.

In [ ]:
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(
    df['Loan_Amount_Term'].mode()[0]
)

In [ ]:
df['Loan_Amount_Term'].isna().sum()

In [ ]:
df['Credit_History'].fillna(1)

## 10. Handling Missing Credit History

`Credit_History` is a binary variable containing 0 and 1.

However, 50 observations have missing credit history. Investigation showed that these missing observations have a different loan approval pattern from observations where `Credit_History = 0`.

Therefore, missing credit history is treated as a separate category rather than being incorrectly classified as either 0 or 1.

The encoding is:

- `1` → Positive credit history
- `0` → No/negative credit history
- `-1` → Credit history not available

In [ ]:
df['Credit_History'] = df['Credit_History'].fillna(-1)

In [ ]:
df['Credit_History'].value_counts()

In [ ]:
df.isna().sum()

In [ ]:
df.shape

In [ ]:
print("Shape:", df.shape)
print("\nTotal missing values:", df.isna().sum().sum())

## 11. Duplicate Record Investigation

Duplicate records can cause problems during machine learning because repeated observations may give the model an unrealistic representation of certain patterns.

Before removing duplicates, we will first determine whether duplicate rows exist and how many there are.

The duplicate investigation will be performed on the cleaned working dataset.

In [ ]:
df.duplicated().sum()

## 12. Data Type and Category Validation

Before exploratory data analysis and machine learning, the structure of each variable is examined.

This step verifies:

- The data type of each variable
- The number of unique values
- The categories present in categorical variables
- Potential inconsistencies in categorical values

Understanding these characteristics will help determine the appropriate preprocessing and encoding techniques for the machine learning models.

In [ ]:
df.info()

In [ ]:
df.nunique().sort_values()

In [ ]:
categorical_cols = [
    'Gender',
    'Married',
    'Dependents',
    'Education',
    'Self_Employed',
    'Property_Area',
    'Loan_Status'
]

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(df[col].value_counts())

In [ ]:
numerical_cols = [
    'ApplicantIncome',
    'CoapplicantIncome',
    'LoanAmount',
    'Loan_Amount_Term',
    'Credit_History'
]

df[numerical_cols].describe()

In [ ]:
df.info()

In [ ]:
df.nunique().sort_values()

# 13. Exploratory Data Analysis

Exploratory Data Analysis (EDA) is performed to understand the distribution of the variables and identify relationships between applicant characteristics and loan approval.

The analysis focuses on:

- Loan approval distribution
- Applicant demographics
- Income distribution
- Loan amount distribution
- Credit history
- Loan amount term
- Education
- Employment status
- Property area
- Dependents
- Relationships between predictor variables and loan approval

The findings from this stage will guide feature engineering and model development.

### 13.1 Loan Approval Distribution

The distribution of the target variable `Loan_Status` is examined to determine whether the dataset is balanced or imbalanced.

In [ ]:
df['Loan_Status'].value_counts()

In [ ]:
df['Loan_Status'].value_counts(normalize=True) * 100

In [ ]:
df['Loan_Status'].value_counts().plot(
    kind='bar',
    figsize=(6, 4), color = ['green', 'red']
)

plt.title('Loan Approval Distribution', loc = 'left', size =20)
plt.xlabel('Loan Status')
plt.ylabel('Number of Applications')
plt.xticks(rotation=0)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(df['ApplicantIncome'], bins=30, color = 'green')

plt.title('Applicant Income Distribution', loc = 'left', size = 20)
plt.xlabel('Applicant Income')
plt.ylabel('Frequency')

plt.show()

In [ ]:
pd.crosstab(
    df['Credit_History'],
    df['Loan_Status']
).plot(
    kind='bar',
    figsize=(7, 5), color = ['red', 'green']
)

plt.title('Credit History vs Loan Status', size = 20, loc = 'left')
plt.xlabel('Credit History')
plt.ylabel('Number of Applications')
plt.xticks(rotation=0)
plt.legend(title='Loan Status')
plt.show()

### 13.3 Education and Loan Approval

The relationship between an applicant's education level and loan approval is examined.

Approval rates are compared between graduate and non-graduate applicants to determine whether education level is associated with loan approval.

In [ ]:
pd.crosstab(
    df['Education'],
    df['Loan_Status'],
    normalize='index'
) * 100

In [ ]:
pd.crosstab(
    df['Education'],
    df['Loan_Status']
)

In [ ]:
education_status = pd.crosstab(
    df['Education'],
    df['Loan_Status']
)

education_status.plot(
    kind='bar',
    figsize=(7, 5), color = ['red', 'green']
)

plt.title('Education vs Loan Status', loc = 'left', size = 20)
plt.xlabel('Education')
plt.ylabel('Number of Applications')
plt.xticks(rotation=0)
plt.legend(title='Loan Status')

plt.show()

### 13.4 Property Area and Loan Approval

The relationship between the applicant's property area and loan approval is examined.

Approval rates are compared across rural, semiurban, and urban property areas to determine whether property location is associated with loan approval.

In [ ]:
pd.crosstab(
    df['Property_Area'],
    df['Loan_Status'],
    normalize='index'
) * 100

In [ ]:
pd.crosstab(
    df['Property_Area'],
    df['Loan_Status']
)

In [ ]:
property_status = pd.crosstab(
    df['Property_Area'],
    df['Loan_Status']
)

property_status.plot(
    kind='bar',
    figsize=(7, 5), color = ['red', 'green']
)

plt.title('Property Area vs Loan Status', loc = 'left', size = 20)
plt.xlabel('Property Area')
plt.ylabel('Number of Applications')
plt.xticks(rotation=0)
plt.legend(title='Loan Status')

plt.show()

pd.crosstab(
    df['Education'],
    df['Loan_Status'],
    normalize='index'
) * 100

### Education and Loan Approval — Finding

Graduate applicants have a higher loan approval rate (70.83%) compared with non-graduate applicants (61.19%).

This represents a difference of approximately 9.64 percentage points. Although education appears to be associated with loan approval in the dataset, this relationship does not imply causation because other applicant characteristics may influence the loan decision.

In [ ]:
plt.figure(figsize=(8, 4))

plt.boxplot(df['ApplicantIncome'])

plt.title('Applicant Income Boxplot', loc = 'left', size = 20)
plt.ylabel('Applicant Income')

plt.show()

In [ ]:
df.groupby('Loan_Status')['ApplicantIncome'].median()

In [ ]:
df.groupby('Loan_Status')['ApplicantIncome'].describe()

In [ ]:
income_approved = df[df['Loan_Status'] == 'Y']['ApplicantIncome']
income_rejected = df[df['Loan_Status'] == 'N']['ApplicantIncome']

plt.figure(figsize=(8, 5))

plt.boxplot(
    [income_approved, income_rejected],
    tick_labels=['Approved', 'Rejected']
)

plt.title('Applicant Income by Loan Status', loc = 'left', size = 20)
plt.xlabel('Loan Status')
plt.ylabel('Applicant Income')

plt.show()

In [ ]:
df.groupby('Loan_Status')['ApplicantIncome'].describe()

In [ ]:
df.groupby('Loan_Status')['ApplicantIncome'].median()

In [ ]:
df.groupby('Loan_Status')['ApplicantIncome'].describe()

### Applicant Income and Loan Approval — Finding

Applicant income shows very similar central tendencies between approved and rejected loan applications.

The median applicant income is 3,833.5 for rejected applications and 3,812.5 for approved applications. The mean income is also similar between the two groups.

However, the income distribution is strongly right-skewed, with a maximum income of 81,000 compared with a median of approximately 3,813.

Rejected applications also show greater income variability than approved applications, as indicated by their higher standard deviation.

Overall, applicant income alone does not appear to strongly distinguish loan approval outcomes. However, its skewed distribution may warrant transformation during feature engineering.

### 13.6 Coapplicant Income Distribution

Coapplicant income is analyzed to understand its distribution, central tendency, variability, and potential extreme observations.

The variable is also compared across loan approval outcomes to determine whether coapplicant income is associated with loan approval.

In [ ]:
df['CoapplicantIncome'].describe()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(df['CoapplicantIncome'], bins=30, color = 'green')

plt.title('Coapplicant Income Distribution', size = 20, loc = 'left')
plt.xlabel('Coapplicant Income')
plt.ylabel('Frequency')

plt.show()

In [ ]:
plt.figure(figsize=(8, 4))

plt.boxplot(df['CoapplicantIncome'])

plt.title('Coapplicant Income Boxplot', size = 20, loc = 'left')
plt.ylabel('Coapplicant Income')

plt.show()

In [ ]:
df.groupby('Loan_Status')['CoapplicantIncome'].describe()

In [ ]:
df.groupby('Loan_Status')['CoapplicantIncome'].median()

In [ ]:
co_income_approved = df[df['Loan_Status'] == 'Y']['CoapplicantIncome']
co_income_rejected = df[df['Loan_Status'] == 'N']['CoapplicantIncome']

plt.figure(figsize=(8, 5))

plt.boxplot(
    [co_income_approved, co_income_rejected],
    tick_labels=['Approved', 'Rejected']
)

plt.title('Coapplicant Income by Loan Status', loc = 'left', size = 20)
plt.xlabel('Loan Status')
plt.ylabel('Coapplicant Income')

plt.show()

### Coapplicant Income and Loan Approval — Finding

Coapplicant income shows a substantial difference in median values between approved and rejected applications.

The median coapplicant income is 1,239.5 for approved applications compared with 268.0 for rejected applications.

The variable is strongly right-skewed, with a large number of applicants having zero coapplicant income and a small number of observations with very high incomes. This causes the mean to differ substantially from the median.

Extreme values are retained at this stage because they may represent legitimate applicants rather than data errors.

Coapplicant income appears to contain potentially useful information for predicting loan approval, although its predictive contribution will be evaluated alongside the other variables during model development.

### 13.7 Loan Amount Distribution

The distribution of loan amounts is examined to understand its central tendency, variability, skewness, and potential extreme observations.

Loan amount is also compared across loan approval outcomes to determine whether the requested loan amount is associated with loan approval.

Extreme values will be investigated rather than automatically removed, since they may represent legitimate loan applications.

In [ ]:
df['LoanAmount'].describe()

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(df['LoanAmount'], bins=30, color = 'green')

plt.title('Loan Amount Distribution', loc = 'left', size = 20)
plt.xlabel('Loan Amount')
plt.ylabel('Frequency')

plt.show()

In [ ]:
plt.figure(figsize=(8, 4))

plt.boxplot(df['LoanAmount'].dropna())

plt.title('Loan Amount Boxplot', loc = 'left', size = 20)
plt.ylabel('Loan Amount')

plt.show()

In [ ]:
df.groupby('Loan_Status')['LoanAmount'].median()

In [ ]:
df.groupby('Loan_Status')['LoanAmount'].describe()

In [ ]:
loan_approved = df[df['Loan_Status'] == 'Y']['LoanAmount']
loan_rejected = df[df['Loan_Status'] == 'N']['LoanAmount']

plt.figure(figsize=(8, 5))

plt.boxplot(
    [loan_approved, loan_rejected],
    tick_labels=['Approved', 'Rejected']
)

plt.title('Loan Amount by Loan Status', loc = 'left', size = 20)
plt.xlabel('Loan Status')
plt.ylabel('Loan Amount')

plt.show()

### Loan Amount and Loan Approval — Finding

The median loan amount is identical for approved and rejected applications at 128.

The mean loan amounts are also relatively similar, with 149.89 for rejected applications and 143.87 for approved applications.

Although the loan amount distribution contains extreme observations, these values have not been removed because they may represent legitimate loan applications.

Overall, loan amount alone does not appear to strongly distinguish approved from rejected applications based on the descriptive statistics. Its predictive contribution will therefore be evaluated together with the other applicant characteristics during model development.

### 13.8 Loan Amount Term and Loan Approval

The relationship between loan repayment term and loan approval is examined.

The distribution of loan terms is analyzed and approval rates are compared across the different repayment periods to determine whether loan term is associated with loan approval.

In [ ]:
df['Loan_Amount_Term'].value_counts().sort_index()

In [ ]:
pd.crosstab(
    df['Loan_Amount_Term'],
    df['Loan_Status']
)

In [ ]:
pd.crosstab(
    df['Loan_Amount_Term'],
    df['Loan_Status'],
    normalize='index'
) * 100

### Loan Amount Term and Loan Approval — Finding

Loan repayment terms are highly concentrated around 360 months, with 526 of the 614 applications having this term.

Among applicants with a 360-month loan term, 69.77% were approved and 30.23% were rejected.

Some less common loan-term categories show extreme approval rates, including 100% approval for 12-, 60-, and 120-month terms. However, these categories contain only 1–3 observations and therefore should not be interpreted as strong evidence of an association.

The 480-month category contains 15 observations and has a lower approval rate of 40%, but the sample size is still relatively small.

Overall, `Loan_Amount_Term` will be retained as a potential predictor, while rare categories will be treated cautiously during model development.

In [ ]:
term_status = pd.crosstab(
    df['Loan_Amount_Term'],
    df['Loan_Status']
)

term_status.plot(
    kind='bar',
    figsize=(10, 5), color = ['red', 'green']
)

plt.title('Loan Amount Term vs Loan Status', loc = 'left', size = 20)
plt.xlabel('Loan Amount Term (Months)')
plt.ylabel('Number of Applications')
plt.xticks(rotation=45)
plt.legend(title='Loan Status')

plt.show()

In [ ]:
pd.crosstab(
    df['Gender'],
    df['Loan_Status'],
    normalize='index'
) * 100

In [ ]:
pd.crosstab(
    df['Married'],
    df['Loan_Status'],
    normalize='index'
) * 100

In [ ]:
pd.crosstab(
    df['Dependents'],
    df['Loan_Status'],
    normalize='index'
) * 100

In [ ]:
pd.crosstab(
    df['Self_Employed'],
    df['Loan_Status'],
    normalize='index'
) * 100

In [ ]:
gender_status = pd.crosstab(df['Gender'], df['Loan_Status'])

gender_status.plot(
    kind='bar',
    figsize=(7, 5), color = ['red', 'green']
)

plt.title('Gender vs Loan Status', loc = 'left', size =20)
plt.xlabel('Gender')
plt.ylabel('Number of Applications')
plt.xticks(rotation=0)
plt.legend(title='Loan Status')

plt.show()

In [ ]:
married_status = pd.crosstab(df['Married'], df['Loan_Status'])

married_status.plot(
    kind='bar',
    figsize=(7, 5), color = ['red', 'green']
)

plt.title('Marital Status vs Loan Status', loc = 'left', size = 20)
plt.xlabel('Married')
plt.ylabel('Number of Applications')
plt.xticks(rotation=0)
plt.legend(title='Loan Status')

plt.show()

In [ ]:
dependents_status = pd.crosstab(df['Dependents'], df['Loan_Status'])

dependents_status.plot(
    kind='bar',
    figsize=(8, 5),color = ['red', 'green']
)

plt.title('Dependents vs Loan Status', loc = 'left', size = 20)
plt.xlabel('Number of Dependents')
plt.ylabel('Number of Applications')
plt.xticks(rotation=0)
plt.legend(title='Loan Status')

plt.show()

In [ ]:
employment_status = pd.crosstab(df['Self_Employed'], df['Loan_Status'])

employment_status.plot(
    kind='bar',
    figsize=(7, 5), color = ['red', 'green']
)

plt.title('Self-Employment Status vs Loan Status', loc = 'left', size = 20)
plt.xlabel('Self-Employed')
plt.ylabel('Number of Applications')
plt.xticks(rotation=0)
plt.legend(title='Loan Status')

plt.show()

### Categorical Features and Loan Approval

The categorical variables Gender, Married, Dependents, and Self-Employed are compared with loan approval status.

Approval rates are calculated within each category to identify potential differences in loan approval outcomes. Bar charts are also used to visually compare approved and rejected applications across the categories.

The analysis helps identify categorical variables that may contain useful predictive information for the machine learning models.

### Categorical Features and Loan Approval — Findings

#### Gender
Loan approval rates are similar for female and male applicants, with approval rates of 66.96% and 69.12%, respectively. This suggests that Gender does not have a strong association with loan approval in the dataset.

#### Married
Married applicants have a higher approval rate (71.82%) compared with applicants who are not married (62.91%). This represents a difference of approximately 8.91 percentage points.

#### Dependents
Approval rates vary across dependent categories. Applicants with two dependents have the highest approval rate (75.25%), while applicants with one or three or more dependents have lower approval rates (64.71%). These differences should be interpreted cautiously because category sizes differ.

#### Self-Employed
Approval rates are almost identical for self-employed (68.29%) and non-self-employed applicants (68.80%). Therefore, Self_Employed does not appear to have a strong association with loan approval based on descriptive analysis alone.